# Automated Notion Task Email Agent with LangGraph and Hugging Face

This tutorial builds a simple agent that:

- Retrieves **completed tasks** from a Notion database
- Retrieves **upcoming tasks for the next week**
- Uses a **Hugging Face model** to generate an email summary
- Runs the workflow using **LangGraph**


## 1. Install Dependencies

In [ ]:
!pip install langgraph transformers torch notion-client python-dotenv

## 2. Import Libraries

In [ ]:
import os
from typing import TypedDict, List
from datetime import datetime, timedelta

from notion_client import Client
from transformers import pipeline

import smtplib
from email.mime.text import MIMEText

from langgraph.graph import StateGraph, END

## 3. Configure Credentials
Replace these placeholders with your actual credentials.

In [ ]:
NOTION_TOKEN = "your_notion_token"
DATABASE_ID = "your_database_id"

EMAIL_USER = "sender@email.com"
EMAIL_PASSWORD = "email_password"
EMAIL_TO = "recipient@email.com"

## 4. Initialize Notion Client

In [ ]:
notion = Client(auth=NOTION_TOKEN)

## 5. Load a Hugging Face Model

In [ ]:
generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    max_length=300
)

## 6. Define Agent State
LangGraph uses a shared state object across nodes.

In [ ]:
class AgentState(TypedDict):
    completed_tasks: List[str]
    upcoming_tasks: List[str]
    email_body: str

## 7. Notion Tool: Fetch Completed Tasks

In [ ]:
def fetch_completed_tasks(state: AgentState):

    response = notion.databases.query(
        database_id=DATABASE_ID,
        filter={
            "property": "Status",
            "status": {"equals": "Completed"}
        }
    )

    tasks = []

    for page in response["results"]:
        title = page["properties"]["Task Name"]["title"]
        if title:
            tasks.append(title[0]["plain_text"])

    state["completed_tasks"] = tasks
    return state

## 8. Notion Tool: Fetch Upcoming Tasks

In [ ]:
def fetch_upcoming_tasks(state: AgentState):

    today = datetime.today().date()
    next_week = today + timedelta(days=7)

    response = notion.databases.query(
        database_id=DATABASE_ID,
        filter={
            "and": [
                {"property": "Due Date", "date": {"on_or_after": today.isoformat()}},
                {"property": "Due Date", "date": {"on_or_before": next_week.isoformat()}}
            ]
        }
    )

    tasks = []

    for page in response["results"]:
        title = page["properties"]["Task Name"]["title"]
        if title:
            tasks.append(title[0]["plain_text"])

    state["upcoming_tasks"] = tasks
    return state

## 9. Email Generation Node

In [ ]:
def generate_email(state: AgentState):

    completed = "\n".join(f"- {t}" for t in state["completed_tasks"])
    upcoming = "\n".join(f"- {t}" for t in state["upcoming_tasks"])

    prompt = f"""
Write a concise professional weekly update email.

Completed Tasks:
{completed}

Upcoming Tasks Next Week:
{upcoming}

Email:
"""

    result = generator(prompt)[0]["generated_text"]

    state["email_body"] = result
    return state

## 10. Email Sender Node

In [ ]:
def send_email(state: AgentState):

    msg = MIMEText(state["email_body"])
    msg["Subject"] = "Weekly Task Update"
    msg["From"] = EMAIL_USER
    msg["To"] = EMAIL_TO

    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
        server.login(EMAIL_USER, EMAIL_PASSWORD)
        server.send_message(msg)

    return state

## 11. Build the LangGraph Workflow

In [ ]:
builder = StateGraph(AgentState)

builder.add_node("fetch_completed", fetch_completed_tasks)
builder.add_node("fetch_upcoming", fetch_upcoming_tasks)
builder.add_node("generate_email", generate_email)
builder.add_node("send_email", send_email)

builder.set_entry_point("fetch_completed")

builder.add_edge("fetch_completed", "fetch_upcoming")
builder.add_edge("fetch_upcoming", "generate_email")
builder.add_edge("generate_email", "send_email")
builder.add_edge("send_email", END)

graph = builder.compile()

## 12. Run the Agent

In [ ]:
initial_state = {
    "completed_tasks": [],
    "upcoming_tasks": [],
    "email_body": ""
}

graph.invoke(initial_state)